# Feature Engineering
The process of creating, transforming, or selecting input variables (features) from raw data to improve the performance of machine learning models. It often involves techniques like scaling, encoding categorical data, creating new features from existing ones, or reducing dimensionality. Essentially, it’s about turning raw data into meaningful inputs that help models learn patterns more effectively.

Create meaningful features:
- One-hot encoding
- Extract date features (weekday, month, hour…)
- Domain logic features
- Combine or derive features (ratios, interactions)
- Transform skew (log transform)
- Text preprocessing (tokenisation, embeddings)

Here we will be using pandas to handle storing and processing the data

In [75]:
import pandas as pd

df = pd.read_csv("data/nba_games.csv",index_col=0)

In [76]:
df["won_opp"] = ~df["won"]

In [77]:
df.tail()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,orb,...,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won,won_opp
25685,240.0,40.0,94.0,0.426,14.0,32.0,0.438,26.0,32.0,19.0,...,32.2,230.0,136.0,IND,109,0,2025,2025-06-16,True,False
25686,240.0,31.0,74.0,0.419,8.0,30.0,0.267,21.0,26.0,4.0,...,28.4,177.0,102.0,IND,108,1,2025,2025-06-19,False,True
25687,240.0,38.0,92.0,0.413,15.0,42.0,0.357,17.0,25.0,11.0,...,44.1,186.0,114.0,OKC,91,0,2025,2025-06-19,True,False
25688,240.0,29.0,70.0,0.414,11.0,28.0,0.393,22.0,29.0,12.0,...,82.8,162.0,114.0,OKC,103,1,2025,2025-06-22,False,True
25689,240.0,35.0,87.0,0.402,11.0,40.0,0.275,22.0,31.0,13.0,...,42.5,134.0,123.0,IND,91,0,2025,2025-06-22,True,False


In [78]:
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.max_colwidth', None,
    'display.width', None
):
    print(df.tail(2))

          mp    fg   fga    fg%    3p   3pa    3p%    ft   fta   orb   drb  \
25688  240.0  29.0  70.0  0.414  11.0  28.0  0.393  22.0  29.0  12.0  33.0   
25689  240.0  35.0  87.0  0.402  11.0  40.0  0.275  22.0  31.0  13.0  27.0   

        trb   ast   stl  blk   tov    pf    pts    ts%   efg%  3par    ftr  \
25688  45.0  17.0   6.0  4.0  23.0  24.0   91.0  0.550  0.493  0.40  0.414   
25689  40.0  20.0  14.0  8.0   8.0  23.0  103.0  0.512  0.466  0.46  0.356   

       orb%  drb%  trb%  ast%  stl%  blk%  tov%   usg%   ortg   drtg  fg_max  \
25688  30.8  71.7  52.9  58.6   6.6   8.5  21.7  100.0   99.6  112.7     8.0   
25689  28.3  69.2  47.1  57.1  15.3  19.0   7.4  100.0  112.7   99.6     8.0   

       fga_max  fg%_max  3p_max  3pa_max  3p%_max  ft_max  fta_max  orb_max  \
25688     14.0    0.615     3.0      5.0    0.750    10.0     10.0      7.0   
25689     27.0    0.750     2.0     12.0    0.667    11.0     12.0      3.0   

       drb_max  trb_max  ast_max  stl_max  blk_max 

In [79]:
df_test = df.sort_values(["team_opp", "date"])
df_test[["team_opp","date"]]

,team_opp,date
12806,ATL,2020-12-23
12832,ATL,2020-12-26
12868,ATL,2020-12-28
12897,ATL,2020-12-30
12933,ATL,2021-01-01
...,...,...
25392,WAS,2025-04-06
25413,WAS,2025-04-08
25425,WAS,2025-04-09
25473,WAS,2025-04-11


### Adding days rested since last game

In [80]:
def set_days_rested(df,team,prev_date,days_rested):
    # Sort within each team by date
    df = df.sort_values([team, "date"])

    # Compute previous game date for each team
    df[prev_date] = df.groupby(team)["date"].shift(1)
    
    #Set the dates as datetime
    df["date"] = pd.to_datetime(df["date"])
    df[prev_date] = pd.to_datetime(df[prev_date])

    # Difference in days
    df[days_rested] = (df["date"] - df[prev_date]).dt.days

    # First game of the season has no previous game → fill with something reasonable
    df[days_rested] = df[days_rested].fillna(0)

    return df

def set_both_teams_days_rested(df):
    df = set_days_rested(df,"team","prev_date","days_rested")
    df = set_days_rested(df,"team_opp","prev_date_opp","days_rested_opp")
    return df

df = set_both_teams_days_rested(df)

In [81]:
df = df.dropna()
df = df.sort_values("date")

In [82]:
df = df[df["days_rested"] <= 4]
df = df[df["days_rested_opp"] <= 4]

In [83]:
df[["team","team_opp","home","season","days_rested","days_rested_opp","won"]].head(20)

,team,team_opp,home,season,days_rested,days_rested_opp,won
12815,DAL,LAL,0,2021,2.0,3.0,False
12810,MIL,GSW,1,2021,2.0,3.0,True
12819,BRK,BOS,0,2021,3.0,2.0,True
12812,MIA,NOP,1,2021,2.0,2.0,True
12816,DEN,LAC,1,2021,2.0,3.0,False
12818,BOS,BRK,1,2021,2.0,3.0,False
12817,LAC,DEN,0,2021,3.0,2.0,True
12811,GSW,MIL,0,2021,3.0,2.0,False
12813,NOP,MIA,0,2021,2.0,2.0,False
12814,LAL,DAL,1,2021,3.0,2.0,True


In [84]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,orb,...,total_opp,home_opp,season,date,won,won_opp,prev_date,days_rested,prev_date_opp,days_rested_opp
12815,240.0,41.0,83.0,0.494,13.0,32.0,0.406,20.0,26.0,3.0,...,138,1,2021,2020-12-25,False,True,2020-12-23,2.0,2020-12-22,3.0
12810,240.0,50.0,90.0,0.556,20.0,36.0,0.556,18.0,26.0,11.0,...,99,0,2021,2020-12-25,True,False,2020-12-23,2.0,2020-12-22,3.0
12819,240.0,41.0,76.0,0.539,15.0,29.0,0.517,26.0,31.0,9.0,...,95,1,2021,2020-12-25,True,False,2020-12-22,3.0,2020-12-23,2.0
12812,240.0,38.0,75.0,0.507,16.0,37.0,0.432,19.0,23.0,4.0,...,98,0,2021,2020-12-25,True,False,2020-12-23,2.0,2020-12-23,2.0
12816,240.0,37.0,81.0,0.457,11.0,36.0,0.306,23.0,31.0,10.0,...,121,0,2021,2020-12-25,False,True,2020-12-23,2.0,2020-12-22,3.0


In [85]:
for col in df.columns:
    print(col)

mp
fg
fga
fg%
3p
3pa
3p%
ft
fta
orb
drb
trb
ast
stl
blk
tov
pf
pts
ts%
efg%
3par
ftr
orb%
drb%
trb%
ast%
stl%
blk%
tov%
usg%
ortg
drtg
fg_max
fga_max
fg%_max
3p_max
3pa_max
3p%_max
ft_max
fta_max
orb_max
drb_max
trb_max
ast_max
stl_max
blk_max
tov_max
pf_max
pts_max
gmsc_max
ts%_max
efg%_max
3par_max
ftr_max
orb%_max
drb%_max
trb%_max
ast%_max
stl%_max
blk%_max
tov%_max
usg%_max
ortg_max
drtg_max
team
total
home
mp_opp
fg_opp
fga_opp
fg%_opp
3p_opp
3pa_opp
3p%_opp
ft_opp
fta_opp
orb_opp
drb_opp
trb_opp
ast_opp
stl_opp
blk_opp
tov_opp
pf_opp
pts_opp
ts%_opp
efg%_opp
3par_opp
ftr_opp
orb%_opp
drb%_opp
trb%_opp
ast%_opp
stl%_opp
blk%_opp
tov%_opp
usg%_opp
ortg_opp
drtg_opp
fg_max_opp
fga_max_opp
fg%_max_opp
3p_max_opp
3pa_max_opp
3p%_max_opp
ft_max_opp
fta_max_opp
orb_max_opp
drb_max_opp
trb_max_opp
ast_max_opp
stl_max_opp
blk_max_opp
tov_max_opp
pf_max_opp
pts_max_opp
gmsc_max_opp
ts%_max_opp
efg%_max_opp
3par_max_opp
ftr_max_opp
orb%_max_opp
drb%_max_opp
trb%_max_opp
ast%_max_opp
stl%_m

## Rolling averages
First useful feature to add to our dataframe is the rolling averages which our model can use to see how each team is performing over their last 10 games

In [86]:
removed_columns = ["season", "date", "team", "team_opp"]
selected_columns = list(df.columns[~df.columns.isin(removed_columns)])
opp_columns = [c for c in selected_columns if "_opp" in c]
team_columns = list(df.columns[~df.columns.isin(removed_columns + opp_columns)])

In [87]:
def find_team_averages(group,n):
    # Compute rolling mean with an n-game window, shifted so current row isn’t included
    return group.shift().rolling(n, min_periods=1).mean()

def get_rolling_df(df,keys,cols_to_roll,n):
    df = df.sort_values([keys[0],"date"])
    df_rolling = find_team_averages(df.groupby(keys, group_keys=False)[cols_to_roll],n)
    rolling_cols = [f"{col}_{n}" for col in df_rolling.columns]
    df_rolling.columns = rolling_cols
    return df_rolling

df_rolling_10 = get_rolling_df(df,["team","season"],team_columns,10)
df_rolling_5 = get_rolling_df(df,["team","season"],team_columns,5)

df_rolling_10_opp = get_rolling_df(df,["team_opp","season"],opp_columns,10)
df_rolling_5_opp = get_rolling_df(df,["team_opp","season"],opp_columns,5)

df = pd.concat([df, df_rolling_5, df_rolling_10, df_rolling_10_opp, df_rolling_5_opp], axis=1)

In [88]:
with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.max_colwidth', None,
    'display.width', None
):
    print(df.tail(2))

          mp    fg   fga    fg%    3p   3pa    3p%    ft   fta   orb   drb  \
25688  240.0  29.0  70.0  0.414  11.0  28.0  0.393  22.0  29.0  12.0  33.0   
25689  240.0  35.0  87.0  0.402  11.0  40.0  0.275  22.0  31.0  13.0  27.0   

        trb   ast   stl  blk   tov    pf    pts    ts%   efg%  3par    ftr  \
25688  45.0  17.0   6.0  4.0  23.0  24.0   91.0  0.550  0.493  0.40  0.414   
25689  40.0  20.0  14.0  8.0   8.0  23.0  103.0  0.512  0.466  0.46  0.356   

       orb%  drb%  trb%  ast%  stl%  blk%  tov%   usg%   ortg   drtg  fg_max  \
25688  30.8  71.7  52.9  58.6   6.6   8.5  21.7  100.0   99.6  112.7     8.0   
25689  28.3  69.2  47.1  57.1  15.3  19.0   7.4  100.0  112.7   99.6     8.0   

       fga_max  fg%_max  3p_max  3pa_max  3p%_max  ft_max  fta_max  orb_max  \
25688     14.0    0.615     3.0      5.0    0.750    10.0     10.0      7.0   
25689     27.0    0.750     2.0     12.0    0.667    11.0     12.0      3.0   

       drb_max  trb_max  ast_max  stl_max  blk_max 

In [89]:
df = df.dropna() #Drop all null/na values

In [90]:
df.describe()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,orb,...,stl%_max_opp_5,blk%_max_opp_5,tov%_max_opp_5,usg%_max_opp_5,ortg_max_opp_5,drtg_max_opp_5,total_opp_5,home_opp_5,won_opp_5,days_rested_opp_5
count,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,...,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000,12310.000000
mean,241.397238,41.393501,88.350853,0.469438,12.721690,35.260195,0.359983,17.262551,22.121608,10.445816,...,6.039667,9.606656,40.088865,35.205241,184.830038,122.509311,112.848455,0.500356,0.503726,2.027326
std,6.347683,5.283804,7.066354,0.055160,3.917362,6.922652,0.083728,5.807592,6.961280,3.839976,...,2.208971,3.610381,10.216076,4.477366,20.212696,5.768240,7.023818,0.227426,0.254906,0.261905
min,240.000000,22.000000,64.000000,0.277000,2.000000,10.000000,0.069000,0.000000,0.000000,0.000000,...,1.700000,2.180000,16.520000,25.620000,131.750000,101.600000,88.600000,0.000000,0.000000,1.400000
25%,240.000000,38.000000,84.000000,0.432000,10.000000,30.000000,0.303000,13.000000,17.000000,8.000000,...,4.700000,7.240000,32.320000,32.180000,170.400000,118.600000,108.000000,0.400000,0.400000,1.800000
50%,240.000000,41.000000,88.000000,0.469000,12.000000,35.000000,0.357000,17.000000,22.000000,10.000000,...,5.580000,8.900000,38.680000,34.400000,183.500000,122.500000,112.800000,0.500000,0.500000,2.000000
75%,240.000000,45.000000,93.000000,0.506000,15.000000,40.000000,0.415000,21.000000,27.000000,13.000000,...,6.740000,11.120000,46.460000,37.225000,197.800000,126.400000,117.800000,0.600000,0.600000,2.200000
max,315.000000,65.000000,121.000000,0.689000,29.000000,63.000000,0.684000,44.000000,53.000000,29.000000,...,30.375000,36.925000,100.000000,66.250000,278.600000,144.800000,136.800000,1.000000,1.000000,3.200000


In [91]:
for col in df.columns:
    print(col)

mp
fg
fga
fg%
3p
3pa
3p%
ft
fta
orb
drb
trb
ast
stl
blk
tov
pf
pts
ts%
efg%
3par
ftr
orb%
drb%
trb%
ast%
stl%
blk%
tov%
usg%
ortg
drtg
fg_max
fga_max
fg%_max
3p_max
3pa_max
3p%_max
ft_max
fta_max
orb_max
drb_max
trb_max
ast_max
stl_max
blk_max
tov_max
pf_max
pts_max
gmsc_max
ts%_max
efg%_max
3par_max
ftr_max
orb%_max
drb%_max
trb%_max
ast%_max
stl%_max
blk%_max
tov%_max
usg%_max
ortg_max
drtg_max
team
total
home
mp_opp
fg_opp
fga_opp
fg%_opp
3p_opp
3pa_opp
3p%_opp
ft_opp
fta_opp
orb_opp
drb_opp
trb_opp
ast_opp
stl_opp
blk_opp
tov_opp
pf_opp
pts_opp
ts%_opp
efg%_opp
3par_opp
ftr_opp
orb%_opp
drb%_opp
trb%_opp
ast%_opp
stl%_opp
blk%_opp
tov%_opp
usg%_opp
ortg_opp
drtg_opp
fg_max_opp
fga_max_opp
fg%_max_opp
3p_max_opp
3pa_max_opp
3p%_max_opp
ft_max_opp
fta_max_opp
orb_max_opp
drb_max_opp
trb_max_opp
ast_max_opp
stl_max_opp
blk_max_opp
tov_max_opp
pf_max_opp
pts_max_opp
gmsc_max_opp
ts%_max_opp
efg%_max_opp
3par_max_opp
ftr_max_opp
orb%_max_opp
drb%_max_opp
trb%_max_opp
ast%_max_opp
stl%_m

In [92]:
df[["team","team_opp","won_10","won_opp_10","date"]]

,team,team_opp,won_10,won_opp_10,date
12810,MIL,GSW,0.444444,0.333333,2020-12-25
12811,GSW,MIL,0.333333,0.444444,2020-12-25
12812,MIA,NOP,0.444444,0.777778,2020-12-25
12813,NOP,MIA,0.777778,0.444444,2020-12-25
12814,LAL,DAL,0.777778,0.555556,2020-12-25
...,...,...,...,...,...
25685,OKC,IND,0.700000,0.600000,2025-06-16
25686,OKC,IND,0.700000,0.500000,2025-06-19
25687,IND,OKC,0.500000,0.700000,2025-06-19
25688,IND,OKC,0.500000,0.700000,2025-06-22


## Difference Features
To reduce the complexity of the features and the potential for noise to reduce model performance, we can change the features to be the difference in stats between the two teams playing instead of inputting each team's performance.

For example, instead of having **"fg%_10"** and **"fg%_opp_10"**, we can just have **"fg%_diff"** where **fg%_diff = fg%_10 - fg%_opp_10**

In [93]:
def get_team_diff(df):
    #Select all rolling columns and remove invalid columns
    selected_columns = [c for c in df.columns if "_10" in c or "_5" in c]
    
    #If a col has _opp, then it also has the same col for the other team
    selected_columns = [c for c in selected_columns if "_opp" in c]
    
    #Create new df
    df_diff_only = pd.DataFrame(index=df.index)
    
    #For each column, if there is another column with the exact string, without the _opp
    for col in selected_columns:
        other_col = col.replace("_opp", "")
        
        #Compute difference for each numeric column and Add to empty df
        if other_col in df.columns:
            df_diff_only[f"{other_col}_diff"] = df[other_col] - df[col]
            
    return df_diff_only

df_diff = get_team_diff(df)

C:\Users\henXc\AppData\Local\Temp/ipykernel_37016/3822363202.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  df_diff_only[f"{other_col}_diff"] = df[other_col] - df[col]


In [94]:
df_diff.head()

,mp_10_diff,fg_10_diff,fga_10_diff,fg%_10_diff,3p_10_diff,3pa_10_diff,3p%_10_diff,ft_10_diff,fta_10_diff,orb_10_diff,...,stl%_max_5_diff,blk%_max_5_diff,tov%_max_5_diff,usg%_max_5_diff,ortg_max_5_diff,drtg_max_5_diff,total_5_diff,home_5_diff,won_5_diff,days_rested_5_diff
12810,2.777778,3.888889,2.777778,0.027111,2.777778,2.777778,0.056111,-5.777778,-7.444444,-0.777778,...,-3.650,2.075,15.275,-2.025,1.25,10.00,4.0,-0.25,0.00,0.25
12811,-2.777778,-3.888889,-2.777778,-0.027111,-2.777778,-2.777778,-0.056111,5.777778,7.444444,0.777778,...,3.650,-2.075,-15.275,2.025,-1.25,-10.00,-4.0,0.25,0.00,-0.25
12812,0.000000,0.888889,6.555556,-0.026222,-0.555556,-0.333333,-0.013000,-1.888889,-3.444444,0.000000,...,-1.600,20.475,24.500,1.900,-27.25,0.00,-12.5,-0.25,-0.25,0.25
12813,0.000000,-0.888889,-6.555556,0.026222,0.555556,0.333333,0.013000,1.888889,3.444444,0.000000,...,1.600,-20.475,-24.500,-1.900,27.25,0.00,12.5,0.25,0.25,-0.25
12814,-2.777778,2.444444,-3.666667,0.043889,-2.888889,-10.222222,0.024333,-7.222222,-8.333333,-1.777778,...,-0.325,0.750,11.975,-2.150,-1.50,7.25,-12.5,0.50,0.00,0.25


In [95]:
for col in df_diff.columns:
    print(col)

mp_10_diff
fg_10_diff
fga_10_diff
fg%_10_diff
3p_10_diff
3pa_10_diff
3p%_10_diff
ft_10_diff
fta_10_diff
orb_10_diff
drb_10_diff
trb_10_diff
ast_10_diff
stl_10_diff
blk_10_diff
tov_10_diff
pf_10_diff
pts_10_diff
ts%_10_diff
efg%_10_diff
3par_10_diff
ftr_10_diff
orb%_10_diff
drb%_10_diff
trb%_10_diff
ast%_10_diff
stl%_10_diff
blk%_10_diff
tov%_10_diff
usg%_10_diff
ortg_10_diff
drtg_10_diff
fg_max_10_diff
fga_max_10_diff
fg%_max_10_diff
3p_max_10_diff
3pa_max_10_diff
3p%_max_10_diff
ft_max_10_diff
fta_max_10_diff
orb_max_10_diff
drb_max_10_diff
trb_max_10_diff
ast_max_10_diff
stl_max_10_diff
blk_max_10_diff
tov_max_10_diff
pf_max_10_diff
pts_max_10_diff
gmsc_max_10_diff
ts%_max_10_diff
efg%_max_10_diff
3par_max_10_diff
ftr_max_10_diff
orb%_max_10_diff
drb%_max_10_diff
trb%_max_10_diff
ast%_max_10_diff
stl%_max_10_diff
blk%_max_10_diff
tov%_max_10_diff
usg%_max_10_diff
ortg_max_10_diff
drtg_max_10_diff
total_10_diff
home_10_diff
won_10_diff
days_rested_10_diff
mp_5_diff
fg_5_diff
fga_5_dif

In [96]:
selected_columns = ["season","date","home","team","home_opp","team_opp","won"]
df = pd.concat([df[selected_columns], df_diff], axis=1)

In [97]:
df.head()

,season,date,home,team,home_opp,team_opp,won,mp_10_diff,fg_10_diff,fga_10_diff,...,stl%_max_5_diff,blk%_max_5_diff,tov%_max_5_diff,usg%_max_5_diff,ortg_max_5_diff,drtg_max_5_diff,total_5_diff,home_5_diff,won_5_diff,days_rested_5_diff
12810,2021,2020-12-25,1,MIL,0,GSW,True,2.777778,3.888889,2.777778,...,-3.650,2.075,15.275,-2.025,1.25,10.00,4.0,-0.25,0.00,0.25
12811,2021,2020-12-25,0,GSW,1,MIL,False,-2.777778,-3.888889,-2.777778,...,3.650,-2.075,-15.275,2.025,-1.25,-10.00,-4.0,0.25,0.00,-0.25
12812,2021,2020-12-25,1,MIA,0,NOP,True,0.000000,0.888889,6.555556,...,-1.600,20.475,24.500,1.900,-27.25,0.00,-12.5,-0.25,-0.25,0.25
12813,2021,2020-12-25,0,NOP,1,MIA,False,0.000000,-0.888889,-6.555556,...,1.600,-20.475,-24.500,-1.900,27.25,0.00,12.5,0.25,0.25,-0.25
12814,2021,2020-12-25,1,LAL,0,DAL,True,-2.777778,2.444444,-3.666667,...,-0.325,0.750,11.975,-2.150,-1.50,7.25,-12.5,0.50,0.00,0.25


### Balancing the dataset 

In [98]:
print(len(df[df['won'] == 1]))
print(len(df[df['won'] == 0]))

6155
6155


In [99]:
from sklearn.utils import resample

def balance_dataframe(df):
    wins = df[df['won'] == 1]
    losses = df[df['won'] == 0]
    
    if len(wins) == len(losses): return df
    if len(wins) > len(losses):
        wins_down = resample(wins,
                   replace=False,
                   n_samples=len(losses),
                   random_state=42)

        return pd.concat([losses, wins_down])
    
    losses_down = resample(losses,
               replace=False,
               n_samples=len(wins),
               random_state=42)
    return pd.concat([wins, losses_down])

df = balance_dataframe(df)

In [100]:
print(len(df[df['won'] == 1]))
print(len(df[df['won'] == 0]))

6155
6155


# Final Dataframe
The final features must only include past data and current information, that can both be inputted into the model.

For the ai model to predict which team is going to win, it needs the recent stats from each team playing.

### The primary key used for finding the most recent game played and pull statistics from that game
- season
- date
- team (home team)
- team_opp (away team)

### Information the model uses to make predictions
- rolling 10-game averages team (contains '_10')
- rolling 10-game averages team_opp (contains '_opp_10')
- rolling 5-game averages team (contains '_5')
- rolling 5-game averages team_opp (contains '_opp_5')
- team performance against team_opp in past 3 games
- team_opp performance against team in past 3 games
- days rested team
- days rested team_opp

### Label/Outcome/Loss/Class
- won   <-- used as label, test model performance with this feature

In [101]:
cols_10 = [col for col in df.columns if "_10" in col]
cols_5 = [col for col in df.columns if "_5" in col]
selected_columns = ["season","date","home","team","team_opp"] + cols_5 + cols_10 + ["won"]

In [102]:
for column in selected_columns:
    print(column)

season
date
home
team
team_opp
mp_5_diff
fg_5_diff
fga_5_diff
fg%_5_diff
3p_5_diff
3pa_5_diff
3p%_5_diff
ft_5_diff
fta_5_diff
orb_5_diff
drb_5_diff
trb_5_diff
ast_5_diff
stl_5_diff
blk_5_diff
tov_5_diff
pf_5_diff
pts_5_diff
ts%_5_diff
efg%_5_diff
3par_5_diff
ftr_5_diff
orb%_5_diff
drb%_5_diff
trb%_5_diff
ast%_5_diff
stl%_5_diff
blk%_5_diff
tov%_5_diff
usg%_5_diff
ortg_5_diff
drtg_5_diff
fg_max_5_diff
fga_max_5_diff
fg%_max_5_diff
3p_max_5_diff
3pa_max_5_diff
3p%_max_5_diff
ft_max_5_diff
fta_max_5_diff
orb_max_5_diff
drb_max_5_diff
trb_max_5_diff
ast_max_5_diff
stl_max_5_diff
blk_max_5_diff
tov_max_5_diff
pf_max_5_diff
pts_max_5_diff
gmsc_max_5_diff
ts%_max_5_diff
efg%_max_5_diff
3par_max_5_diff
ftr_max_5_diff
orb%_max_5_diff
drb%_max_5_diff
trb%_max_5_diff
ast%_max_5_diff
stl%_max_5_diff
blk%_max_5_diff
tov%_max_5_diff
usg%_max_5_diff
ortg_max_5_diff
drtg_max_5_diff
total_5_diff
home_5_diff
won_5_diff
days_rested_5_diff
mp_10_diff
fg_10_diff
fga_10_diff
fg%_10_diff
3p_10_diff
3pa_10_di

In [103]:
df = df.sort_values("date")

In [104]:
df[selected_columns].tail()

,season,date,home,team,team_opp,mp_5_diff,fg_5_diff,fga_5_diff,fg%_5_diff,3p_5_diff,...,blk%_max_10_diff,tov%_max_10_diff,usg%_max_10_diff,ortg_max_10_diff,drtg_max_10_diff,total_10_diff,home_10_diff,won_10_diff,days_rested_10_diff,won
25685,2025,2025-06-16,1,OKC,IND,0.0,3.4,3.2,0.0236,-0.8,...,-0.82,-10.51,4.08,46.4,-1.6,2.3,-0.1,0.1,0.1,True
25686,2025,2025-06-19,0,OKC,IND,0.0,0.4,1.4,-0.0014,-1.4,...,2.22,-11.74,4.07,48.5,-2.7,5.1,0.0,0.2,0.1,False
25687,2025,2025-06-19,1,IND,OKC,0.0,-0.4,-1.4,0.0014,1.4,...,-2.22,11.74,-4.07,-48.5,2.7,-5.1,0.0,-0.2,-0.1,True
25688,2025,2025-06-22,0,IND,OKC,0.0,1.0,2.8,-0.0034,2.2,...,-4.86,13.26,-1.80,-48.2,2.6,-4.1,0.1,-0.2,-0.1,False
25689,2025,2025-06-22,1,OKC,IND,0.0,-1.0,-2.8,0.0034,-2.2,...,4.86,-13.26,1.80,48.2,-2.6,4.1,-0.1,0.2,0.1,True


In [105]:
df = df[selected_columns]

In [106]:
df.to_pickle('data/df_nba_games.pkl')